In [ ]:
import os
import gc
import argparse
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import albumentations as A
from collections import OrderedDict
import matplotlib.pyplot as plt
import seaborn as sns

import sys 
sys.path.append('main-codes/Mamba-UNet-main/code')
from networks.vision_transformer import SwinUnet as ViT_seg
from networks.vision_mamba import MambaUnet as VIM_seg

sys.path.append("main-codes/modified_segmentation_models-main") 
import segmentation_models_pytorch as smp 

sys.path.append("main-codes/HybridModels")
from HybridSwinUNetC import HybridSwinUNetC
from HybridMambaUNetC import HybridMambaUNetC

sys.path.append('main-codes/CKA_Calculator_main')
from main_code import CKA

from config import get_config

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
torch.cuda.empty_cache()

In [ ]:
class MultiClassSegDataset(Dataset):
    
    def __init__(self, df, transform=None,):
        self.df = df
        self.transform = transform
    
    def __getitem__(self, idx):
        
        image_name = self.df.iloc[idx, 1]
        mask_name = self.df.iloc[idx, 2]
        image = cv2.imread(image_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_name, cv2.IMREAD_UNCHANGED)
        image = image.astype('uint8')
        mask = mask[:,:,0]
        if(self.transform is not None):
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        image = image.permute(2, 0, 1)
        image = image.float()/255
        mask = mask.long()
        return image, mask  
        
    def __len__(self):
        return len(self.df)

In [ ]:
INTPUT_DIR = "" # Path to the landcover.ai dataset
CLASSES = ['background', 'building', 'woodlands', 'water', 'road']

testSet = pd.read_csv(INTPUT_DIR + "test.txt", header=None, names=["file"])
testSet["img"] = INTPUT_DIR + "output/" + testSet['file'] + ".jpg"
testSet["mask"] = INTPUT_DIR + "output/" + testSet['file'] + "_m.png"

print("Testing dataset: " + str(len(testSet)))

test_transform = A.Compose(
    [A.CenterCrop(height=224, width=224),])

testDF = testSet.sample(n=1600, random_state=42)

testDS = MultiClassSegDataset(testDF, transform=test_transform)

testDL = DataLoader(testDS, batch_size=8, shuffle=False,
                    num_workers=0, pin_memory=False, drop_last=False)

print("Testing dataset: " + str(len(testDS)))

In [ ]:
# Argument Parser for Mamba-based models
mamba_parser = argparse.ArgumentParser()
mamba_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
mamba_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised', help='experiment_name')
mamba_parser.add_argument('--model', type=str, default='mambaunet', help='model_name')
mamba_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')

mamba_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml", help='path to config file')
mamba_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs.")

mamba_parser.add_argument('--zip', action='store_true', help='use zipped dataset instead of folder dataset')
mamba_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'])
mamba_parser.add_argument('--resume', help='resume from checkpoint')
mamba_parser.add_argument('--accumulation-steps', type=int)
mamba_parser.add_argument('--use-checkpoint', action='store_true')
mamba_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'])
mamba_parser.add_argument('--tag')
mamba_parser.add_argument('--eval', action='store_true')
mamba_parser.add_argument('--throughput', action='store_true')

mamba_parser.add_argument('--max_iterations', type=int, default=25)
mamba_parser.add_argument('--batch_size', type=int, default=8)
mamba_parser.add_argument('--deterministic', type=int, default=1)
mamba_parser.add_argument('--base_lr', type=float, default=0.0001)
mamba_parser.add_argument('--patch_size', type=int, nargs=2, default=[224, 224])
mamba_parser.add_argument('--seed', type=int, default=1337)
mamba_parser.add_argument('--labeled_num', type=int, default=140)

# Step 2: Simulated args for Jupyter/IPython
mamba_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised',
    '--model', 'mambaunet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml',
    '--patch_size', '224', '224',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
mamba_args = mamba_parser.parse_args(args=mamba_simulated_args)

# Step 3: Load config
mamba_config = get_config(mamba_args)

In [ ]:
# Argument Parser for Swin-based models
swin_parser = argparse.ArgumentParser()
swin_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
swin_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised_ViT', help='experiment_name')
swin_parser.add_argument('--model', type=str, default='unet', help='model_name')
swin_parser.add_argument('--max_iterations', type=int, default=25, help='maximum epoch number to train')
swin_parser.add_argument('--batch_size', type=int, default=8, help='batch_size per gpu')
swin_parser.add_argument('--deterministic', type=int, default=1, help='whether use deterministic training')
swin_parser.add_argument('--base_lr', type=float, default=0.0001, help='segmentation network learning rate')
swin_parser.add_argument('--patch_size', type=list, default=[224, 224], help='patch size of network input')
swin_parser.add_argument('--seed', type=int,  default=1337, help='random seed')
swin_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')
swin_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml", help='path to config file', )
swin_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs. ")
swin_parser.add_argument('--zip', action='store_true',
                    help='use zipped dataset instead of folder dataset')
swin_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'],
                    help='no: no cache, '
                         'full: cache all data, '
                         'part: sharding the dataset into nonoverlapping pieces and only cache one piece')
swin_parser.add_argument('--resume', help='resume from checkpoint')
swin_parser.add_argument('--accumulation-steps', type=int,
                    help="gradient accumulation steps")
swin_parser.add_argument('--use-checkpoint', action='store_true',
                    help="whether to use gradient checkpointing to save memory")
swin_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'],
                    help='mixed precision opt level, if O0, no amp is used')
swin_parser.add_argument('--tag', help='tag of experiment')
swin_parser.add_argument('--eval', action='store_true',
                    help='Perform evaluation only')
swin_parser.add_argument('--throughput', action='store_true',
                    help='Test throughput only')

# label and unlabel
swin_parser.add_argument('--labeled_bs', type=int, default=12,
                    help='labeled_batch_size per gpu')
swin_parser.add_argument('--labeled_num', type=int, default=140,
                    help='labeled data')
# costs
swin_parser.add_argument('--ema_decay', type=float,  default=0.99, help='ema_decay')
swin_parser.add_argument('--consistency_type', type=str,
                    default="mse", help='consistency_type')
swin_parser.add_argument('--consistency', type=float,
                    default=0.1, help='consistency')
swin_parser.add_argument('--consistency_rampup', type=float,
                    default=200.0, help='consistency_rampup')

# Step 2: Simulated args for Jupyter/IPython
swin_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised_ViT',
    '--model', 'unet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml',
    '--patch_size', '[224, 224]',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
swin_args = swin_parser.parse_args(args=swin_simulated_args)

# Step 3: Load config
swin_config = get_config(swin_args)

In [ ]:
def calPipe(modelname1, modelname2):

    # Free the models/hooks of the previous call (model -> hook -> CKA -> model is a reference cycle,
    # so they are not released when calPipe returns) -> fixes the CUDA OOM after several calls.
    gc.collect()
    torch.cuda.empty_cache()

    # Models
    SWINUNET = ViT_seg(swin_config, 
                       img_size=swin_args.patch_size,
                       num_classes=swin_args.num_classes).to(device)
    
    SWINUNETC = HybridSwinUNetC(swin_config, 
                                img_size=swin_args.patch_size,
                                num_classes=swin_args.num_classes).to(device)
    
    MAMBAUNET = VIM_seg(mamba_config, 
                        img_size=mamba_args.patch_size,
                        num_classes=mamba_args.num_classes).to(device)
    
    MAMBAUNETC = HybridMambaUNetC(mamba_config, 
                                  img_size=mamba_args.patch_size, 
                                  num_classes=mamba_args.num_classes).to(device)
    
    UNET = smp.Unet(
        encoder_name = "resnet34",                      
        encoder_weights = "imagenet",                   
        decoder_channels = (768, 768, 384, 192, 96),
        in_channels = 3,                                
        classes = 5).to(device)

    # Layers
    SwinUNet_layers = [
        "swin_unet.layers.0.blocks.1.mlp.fc2",     # Swin Block 1
        "swin_unet.layers.1.blocks.1.mlp.fc2",     # Swin Block 2
        "swin_unet.layers.2.blocks.1.mlp.fc2",     # Swin Block 3
        "swin_unet.layers.3.blocks.1.mlp.fc2",     # Bottleneck 
        "swin_unet.layers_up.1.blocks.1.mlp.fc2",  # Swin Block 1
        "swin_unet.layers_up.2.blocks.1.mlp.fc2",  # Swin Block 2
        "swin_unet.layers_up.3.blocks.1.mlp.fc2",  # Swin Block 3
        "swin_unet.up.norm",                       # Final Output
    ]

    SwinUNetC_layers = [
            "encoder.layers.0.blocks.1.mlp.fc2",     # Swin Block 1
            "encoder.layers.1.blocks.1.mlp.fc2",     # Swin Block 2
            "encoder.layers.2.blocks.1.mlp.fc2",     # Swin Block 3
            "encoder.layers.3.blocks.1.mlp.fc2",     # Bottleneck 
            "decoder.decoder1.5",                    # Decoder Block 1
            "decoder.decoder2.5",                    # Decoder Block 2
            "decoder.decoder3.5",                    # Decoder Block 3
            "decoder.decoder4.5",                    # Final Output
        ]

    MambaUNet_layers = [
        "mamba_unet.layers.0.blocks.1.self_attention.out_proj",     # VSS Block 1
        "mamba_unet.layers.1.blocks.1.self_attention.out_proj",     # VSS Block 2
        "mamba_unet.layers.2.blocks.1.self_attention.out_proj",     # VSS Block 3
        "mamba_unet.layers.3.blocks.1.self_attention.out_proj",     # Bottleneck
        "mamba_unet.layers_up.1.blocks.1.self_attention.out_proj",  # VSS Block 1
        "mamba_unet.layers_up.2.blocks.1.self_attention.out_proj",  # VSS Block 2
        "mamba_unet.layers_up.3.blocks.1.self_attention.out_proj",  # VSS Block 3
        "mamba_unet.up.norm"                                        # Final Output 
    ]

    MambaUNetC_layers = [
        "encoder.layers.0.blocks.1.self_attention.out_proj",     # VSS Block 1
        "encoder.layers.1.blocks.1.self_attention.out_proj",     # VSS Block 2
        "encoder.layers.2.blocks.1.self_attention.out_proj",     # VSS Block 3
        "encoder.layers.3.blocks.1.self_attention.out_proj",     # Bottleneck 
        "decoder.decoder1.5",                                    # Decoder Block 1
        "decoder.decoder2.5",                                    # Decoder Block 2
        "decoder.decoder3.5",                                    # Decoder Block 3
        "decoder.decoder4.5"                                     # Final Output
    ]

    UNet_layers = [
        "encoder.layer1.2.bn2",          # Encoder Block 1
        "encoder.layer2.3.bn2",          # Encoder Block 2
        "encoder.layer3.5.bn2",          # Encoder Block 3
        "encoder.layer4.2.bn2",          # Encoder Block 4
        "decoder.blocks.0.conv2.2",      #Bottleneck
        "decoder.blocks.1.conv2.2",      # Decoder Block 1
        "decoder.blocks.2.conv2.2",      # Decoder Block 2
        "decoder.blocks.3.conv2.2",      # Decoder Block 3
        "decoder.blocks.4.conv2.2"       # Decoder Block 4
    ]

    MODELS_DIR = "" #Path to saved models

    if modelname1 == "SwinUNet":
         model1 = SWINUNET
         model1layers = SwinUNet_layers
         checkpoint1 = torch.load(MODELS_DIR + "Best_SwinUNet.pt")
         model1_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",  
                        "Final Output"]
    elif modelname1 == "SwinUNetC": 
         model1 = SWINUNETC
         model1layers = SwinUNetC_layers
         checkpoint1 = torch.load(MODELS_DIR + "Best_SwinUNetC.pt")
         model1_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",  
                        "Final Output"]
    elif modelname1 == "MambaUNet": 
         model1 = MAMBAUNET
         model1layers = MambaUNet_layers
         checkpoint1 = torch.load(MODELS_DIR + "Best_MambaUNet.pt")
         model1_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3", 
                        "Final Output"]
    elif modelname1 == "MambaUNetC": 
         model1 = MAMBAUNETC
         model1layers = MambaUNetC_layers
         checkpoint1 = torch.load(MODELS_DIR + "Best_MambaUNetC.pt")
         model1_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3", 
                        "Final Output"]
    else:
         model1 = UNET
         model1layers = UNet_layers
         checkpoint1 = torch.load(MODELS_DIR + "Best_UNet.pt")
         model1_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", "Encoder Block 4",
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",
                        "Final Output"]

    # Remove `module.` prefix from keys
    new_state_dict1 = OrderedDict()
    for k1, v1 in checkpoint1.items():
        new_key1 = k1.replace("module.", "")  # Remove "module." from the key
        new_state_dict1[new_key1] = v1
    print(model1.load_state_dict(new_state_dict1))
    model1.eval()

    if modelname2 == "SwinUNet":
         model2 = SWINUNET
         model2layers = SwinUNet_layers
         checkpoint2 = torch.load(MODELS_DIR + "Best_SwinUNet.pt")
         model2_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",  
                        "Final Output"]
    elif modelname2 == "SwinUNetC": 
         model2 = SWINUNETC
         model2layers = SwinUNetC_layers
         checkpoint2 = torch.load(MODELS_DIR + "Best_SwinUNetC.pt")
         model2_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",  
                        "Final Output"]
    elif modelname2 == "MambaUNet": 
         model2 = MAMBAUNET
         model2layers = MambaUNet_layers
         checkpoint2 = torch.load(MODELS_DIR + "Best_MambaUNet.pt")
         model2_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3", 
                        "Final Output"]
    elif modelname2 == "MambaUNetC": 
         model2 = MAMBAUNETC
         model2layers = MambaUNetC_layers
         checkpoint2 = torch.load(MODELS_DIR + "Best_MambaUNetC.pt")
         model2_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", 
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3", 
                        "Final Output"]
    else:
         model2 = UNET
         model2layers = UNet_layers
         checkpoint2 = torch.load(MODELS_DIR + "Best_UNet.pt")
         model2_axes = ["Encoder Block 1", "Encoder Block 2", "Encoder Block 3", "Encoder Block 4",
                        "Bottleneck Block", 
                        "Decoder Block 1", "Decoder Block 2", "Decoder Block 3",
                        "Final Output"]

    # Remove `module.` prefix from keys
    new_state_dict2 = OrderedDict()
    for k2, v2 in checkpoint2.items():
        new_key2 = k2.replace("module.", "")  # Remove "module." from the key
        new_state_dict2[new_key2] = v2
    print(model2.load_state_dict(new_state_dict2))
    model2.eval()


    cka = CKA(model1, model2,
              model1_name = modelname1,
              model2_name = modelname2,   
              model1_layers = model1layers,
              model2_layers = model2layers, 
              device='cuda')
    
    cka.compare(testDL)

    cka_matrix = cka.hsic_matrix.cpu().numpy()

    # Double-check shape match
    assert cka_matrix.shape == (len(model1_axes), len(model2_axes)), \
        f"Shape mismatch: CKA shape {cka_matrix.shape}, labels {len(model1_axes)} x {len(model2_axes)}"

    OUT_DIR = "" # Path to save plots

    plt.figure(figsize=(10, 10))

    # 🔥 Better heatmap
    sns.set(font_scale=1.2)  # Make fonts larger
    sns.heatmap(
        cka_matrix,
        xticklabels=model2_axes,
        yticklabels=model1_axes,
        cmap=sns.cubehelix_palette(as_cmap=True),
        annot=True,
        fmt=".2f",
        square=True,
        cbar_kws={"label": "CKA Similarity"},
        linewidths=0.3,
        linecolor='gray'
    )

    # 🖼 Titles and axis labels
    plt.title(f"CKA Similarity: {cka.model1_info['Name']} vs {cka.model2_info['Name']}", fontsize=20, pad=20)
    plt.xlabel(f"{cka.model2_info['Name']} Layers", fontsize=16, labelpad=10)
    plt.ylabel(f"{cka.model1_info['Name']} Layers", fontsize=16, labelpad=10)

    # 🧭 Ticks and layout
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    plt.tight_layout()
    plt.savefig(
         os.path.join(OUT_DIR, f"CKA_{cka.model1_info['Name']}_vs_{cka.model2_info['Name']}.png"),
         dpi=300,
         bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
arch = ["UNet", "MambaUNetC", "SwinUNetC", "MambaUNet", "SwinUNet"]

for i in range(0, len(arch)):
    for j in range(i+1, len(arch)):
        calPipe(arch[i], arch[j])